In [ ]:
# Import Block
!pip install qiskit
!pip install qiskit_aer
import random
import math
import numpy as np
import qiskit as qc
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.7 MB/s eta 0:00:00


In [ ]:
# Internal Function Definitions
def DimensionGiver(n):
  N = n
  Possible_Pairs = []
  for i in range(2,N):
    if (N % i == 0):
      Possible_Pairs.append([i, int(N / i)])
  Dimension = random.choice(Possible_Pairs)
  return Dimension


def KeyChoice(dimension):
  x, y = dimension
  key_x = random.randint(1,x)
  key_y = random.randint(1,y)
  return [key_x, key_y]


def GiveQubitCount(dimension):
  x, y = dimension
  QubitCount = [int(np.ceil(np.log2(x))),int(np.ceil(np.log2(y)))]
  return QubitCount

In [ ]:
# Automated Space Preparation
def Initiate():
  NumberChoices = [4, 6, 9, 10, 14, 15, 21, 22, 25, 26, 33, 34, 35, 38, 39, 46, 49, 51, 55, 57, 58, 62, 65, 69, 74, 77, 82, 85, 86, 87, 91, 93, 94, 95, 106, 111, 115, 118, 119, 121, 122, 123, 129, 133, 134, 141, 142, 143, 145, 146, 155, 158, 159, 161, 166, 169, 177, 178, 183, 185, 187, 194, 201, 202, 203, 205, 206, 209, 213, 214, 215, 217, 218, 219, 221, 226, 235, 237, 247, 249, 253, 254]
  N = random.choice(NumberChoices)
  Dimension = DimensionGiver(N)
  print(f"Dimension: {Dimension}")
  print(f"Key : {KeyChoice(Dimension)}")
  register_A = QuantumRegister(GiveQubitCount(Dimension)[0], name='Width')
  register_B = QuantumRegister(GiveQubitCount(Dimension)[1], name='Height')
  circuit = QuantumCircuit(register_A, register_B)
  print(f"Given N: {N}")
  return circuit, N


#Here, build unitary matrix
def BuildUnitary(Gamma, N):
  n = int(np.ceil(np.log2(N)))
  M = 2**n

  U = np.zeros((M, M))

  for y in range(M):
    if (y < N):
      output = (Gamma * y) % N
    else:
      output = y
    U[output][y] = 1
  return Operator(U)


#The function which runs iterations of unitary until the invariant is found
def RunIter(n,U,N,Y):

  Y_int = int(Y,2)
  print(f"Y = {Y_int}")

  qc = QuantumCircuit(n)
  for i in range(len(Y)):
          if Y[i] == '1':
              qc.x(len(Y) - 1 - i)

  for i in range(N):

    qc.unitary(U, range(n))
    Y_new = Statevector.from_instruction(qc)
    if ((np.argmax(np.abs(Y_new.data)) == Y_int) & (i > 1)):
      print(f"The Value of Invariant is: {i + 1}")
      return (i+1)


#The master Call function which, given a total dimension N, gives the dimensions of the space
def ProvideDimensions(N):
    n = int(np.ceil(np.log2(N)))

    valid_Gamma = []
    for i in range(2, N):
        if math.gcd(i, N) == 1:
            valid_Gamma.append(i)

    Gamma = random.choice(valid_Gamma)
    U = BuildUnitary(Gamma, N)

    Y_val = random.randint(1, N)
    Y = format(Y_val, 'b')

    r = RunIter(n, U, N, Y)
    if r is None:
        return None

    x = pow(int(Gamma), r // 2, int(N))
    if x == 1:
        return None

    Final1 = math.gcd(N, int(x + 1))
    Final2 = math.gcd(N, int(x - 1))
    for f in (Final1, Final2):
        if f > 1 and N % f == 0:
            other = N // f
            if other > 1:
                return (f, other)

    return None



In [ ]:
Space, N = Initiate()

dims = None
while dims is None:
    dims = ProvideDimensions(N)
Dim1, Dim2 = dims
print(f"Dimension 1: {Dim1}")
print(f"Dimension 2: {Dim2}")




Dimension: [7, 13]
Key : [7, 2]
Given N: 91
Y = 8
The Value of Invariant is: 6
Dimension 1: 7
Dimension 2: 13


In [ ]:
import matplotlib.pyplot as plt
import collections

# ── configuration ──────────────────────────────────────────────────────────────
TRIALS_PER_N = 20
BINS = [(1,50),(51,100),(101,150),(151,200),(201,250),(251,300)]
BIN_LABELS = ["1–50","51–100","101–150","151–200","201–250","251–300"]

NumberChoices = [9,   21,    # 1–50
                 51,  57,    # 51–100
                 106, 111,   # 101–150
                 155, 158,   # 151–200
                 201, 205,   # 201–250
                 253, 254,   # 251–300
                 ]
# ── collect results ─────────────────────────────────────────────────────────────
bin_correct = collections.defaultdict(int)
bin_total   = collections.defaultdict(int)

for n_val in NumberChoices:
    target_bin = None
    for (lo, hi) in BINS:
        if lo <= n_val <= hi:
            target_bin = (lo, hi)
            break
    if target_bin is None:
        continue

    for _ in range(TRIALS_PER_N):
        bin_total[target_bin] += 1
        try:
            result = ProvideDimensions(n_val)
        except Exception:
            result = None
        if result is not None:
            f1, f2 = result
            if f1 * f2 == n_val and f1 > 1 and f2 > 1:
                bin_correct[target_bin] += 1

    print(f"N={n_val} done — {bin_correct[target_bin]}/{bin_total[target_bin]} correct so far in bin {target_bin}")

# ── compute percentages ─────────────────────────────────────────────────────────
percentages = []
for b in BINS:
    total   = bin_total[b]
    correct = bin_correct[b]
    pct     = (correct / total * 100) if total > 0 else 0
    percentages.append(pct)
    print(f"N in {BIN_LABELS[BINS.index(b)]}: {correct}/{total} correct  →  {pct:.1f}%")

# ── plot ────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

bar_colors = ["#4C72B0","#DD8452","#55A868","#C44E52","#8172B2"]
bars = ax.bar(BIN_LABELS, percentages, color=bar_colors, edgecolor="white",
              linewidth=0.8, width=0.55)

for bar, pct in zip(bars, percentages):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.2,
            f"{pct:.1f}%",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylim(0, 115)
ax.set_xlabel("Range of N", fontsize=12, labelpad=8)
ax.set_ylabel("Correct Factorisation (%)", fontsize=12, labelpad=8)
ax.set_title("Factorisation Success Rate by N Range",
             fontsize=13, fontweight="bold", pad=12)
ax.axhline(100, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)
ax.spines[["top","right"]].set_visible(False)
ax.yaxis.set_tick_params(labelsize=10)
ax.xaxis.set_tick_params(labelsize=10)

plt.tight_layout()
plt.savefig("factorisation_success_rate.png", dpi=150)
plt.show()

Y = 2
The Value of Invariant is: 3
Y = 9
The Value of Invariant is: 3
Y = 2
The Value of Invariant is: 4
Y = 2
The Value of Invariant is: 3
Y = 2
The Value of Invariant is: 6
Y = 1
The Value of Invariant is: 3
Y = 3
The Value of Invariant is: 4
Y = 5
The Value of Invariant is: 3
Y = 7
The Value of Invariant is: 3
Y = 4
The Value of Invariant is: 4
Y = 3
The Value of Invariant is: 3
Y = 6
The Value of Invariant is: 3
Y = 8
The Value of Invariant is: 4
Y = 4
The Value of Invariant is: 3
Y = 9
The Value of Invariant is: 3
Y = 3
The Value of Invariant is: 4
Y = 2
The Value of Invariant is: 4
Y = 5
The Value of Invariant is: 3
Y = 4
The Value of Invariant is: 3
Y = 8
The Value of Invariant is: 3
N=9 done — 14/20 correct so far in bin (1, 50)
Y = 18
The Value of Invariant is: 4
Y = 5
The Value of Invariant is: 6
Y = 5
The Value of Invariant is: 4
Y = 1
The Value of Invariant is: 6
Y = 4
The Value of Invariant is: 4
Y = 16
The Value of Invariant is: 6
Y = 3
The Value of Invariant is: 6
Y = 3
